# 🦗 01 — Explore Dataset
ดูโครงสร้างข้อมูล, waveform, และ mel-spectrogram เบื้องต้น

**เป้าหมาย:** ทำความเข้าใจ InsectSet32 ก่อนเริ่ม feature extraction


In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
import pandas as pd
from tqdm.notebook import tqdm
from cricket_perception.audio_utils import load_audio, get_duration

plt.style.use("dark_background")
print("✅ Imports OK")


## 1. ตรวจสอบไฟล์ใน Dataset

In [ ]:
DATASET_DIR = Path("../dataset/insectset32/Orthoptera/Orthoptera")
audio_files = sorted(DATASET_DIR.rglob("*.wav")) + sorted(DATASET_DIR.rglob("*.flac"))
print(f"Total audio files: {len(audio_files)}")

# Build a metadata DataFrame
rows = []
for p in tqdm(audio_files[:100], desc="Scanning"):  # scan first 100
    y, sr = load_audio(p)
    rows.append({"file": p.name, "species": p.parent.name,
                 "duration_s": get_duration(y, sr), "sr": sr})

df = pd.DataFrame(rows)
print(f"
Species found: {df.species.nunique()}")
df.groupby("species").size().sort_values(ascending=False).head(15)


## 2. Waveform & Mel-Spectrogram ของตัวอย่าง

In [ ]:
# Pick a random file
import random
random.seed(42)
sample_path = random.choice(audio_files)
y, sr = load_audio(sample_path)
print(f"File: {sample_path.name}")
print(f"Species: {sample_path.parent.name}")
print(f"Duration: {get_duration(y, sr):.2f}s | SR: {sr} Hz")

# ── Plot ──
fig = plt.figure(figsize=(14, 8), facecolor="#0f0f1a")
gs = gridspec.GridSpec(2, 2, figure=fig)

# Waveform
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor("#0f0f1a")
librosa.display.waveshow(y, sr=sr, ax=ax1, color="#7c85ff")
ax1.set_title(f"Waveform — {sample_path.parent.name}/{sample_path.name}",
              color="white")

# Mel-Spectrogram
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor("#0f0f1a")
img = librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel",
                                ax=ax2, cmap="magma")
fig.colorbar(img, ax=ax2, format="%+2.0f dB")
ax2.set_title("Mel-Spectrogram", color="white")

# MFCC
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor("#0f0f1a")
img2 = librosa.display.specshow(mfcc, x_axis="time", ax=ax3, cmap="coolwarm")
fig.colorbar(img2, ax=ax3)
ax3.set_title("MFCC (13 coeff)", color="white")

plt.tight_layout()
plt.savefig("../results/01_explore_sample.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## 3. ดูการกระจายความยาวไฟล์

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="#0f0f1a")
for ax in axes:
    ax.set_facecolor("#151525")
    ax.tick_params(colors="white")

axes[0].hist(df.duration_s, bins=30, color="#7c85ff", edgecolor="none")
axes[0].set_title("Duration Distribution", color="white")
axes[0].set_xlabel("seconds", color="white")

species_counts = df.species.value_counts().head(20)
axes[1].barh(species_counts.index, species_counts.values, color="#ff6e9c")
axes[1].set_title("Top 20 Species by File Count", color="white")
axes[1].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig("../results/01_dataset_stats.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("
📊 Stats:")
print(df.duration_s.describe().round(2))
